In [1]:
# Import thư viện
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import re
import os
from tqdm import tqdm

# Selenium để render JavaScript
try:
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.chrome.service import Service
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    SELENIUM_AVAILABLE = True
    print("✅ Selenium đã được import thành công")
except ImportError:
    SELENIUM_AVAILABLE = False
    print("⚠️ Selenium chưa được cài đặt. Chạy: pip install selenium webdriver-manager")

✅ Selenium đã được import thành công


In [2]:
# Cấu hình
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7',
}

BASE_URL = 'https://monngonmoingay.com'
MIN_DELAY = 0.5
MAX_DELAY = 1.5

# Khởi tạo Selenium driver (headless Chrome)
def init_driver():
    """Khởi tạo Chrome driver ở chế độ headless"""
    if not SELENIUM_AVAILABLE:
        return None
    
    try:
        from webdriver_manager.chrome import ChromeDriverManager
        
        chrome_options = Options()
        chrome_options.add_argument("--headless")  # Chạy ẩn
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        chrome_options.add_argument("--disable-gpu")
        chrome_options.add_argument("--window-size=1920,1080")
        chrome_options.add_argument(f"user-agent={HEADERS['User-Agent']}")
        
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=chrome_options)
        driver.set_page_load_timeout(30)
        
        print("✅ Chrome driver đã khởi tạo thành công")
        return driver
    except Exception as e:
        print(f"❌ Lỗi khởi tạo driver: {e}")
        return None

# Biến global để giữ driver
driver = None

## 1. Lấy danh sách URLs từ WordPress REST API

In [3]:
def get_all_recipe_urls_from_api():
    """
    Lấy tất cả URLs công thức từ WordPress REST API
    API endpoint: /wp-json/wp/v2/monan
    """
    all_urls = []
    page = 1
    per_page = 100  # Maximum allowed by WordPress
    
    print("📡 Fetching recipe URLs from WordPress API...")
    
    while True:
        try:
            api_url = f"https://monngonmoingay.com/wp-json/wp/v2/monan?per_page={per_page}&page={page}"
            response = requests.get(api_url, headers=HEADERS, timeout=30)
            
            if response.status_code == 400:
                print(f"   ✓ Reached end at page {page}")
                break
            
            if response.status_code != 200:
                print(f"   ✗ Error at page {page}: {response.status_code}")
                break
            
            data = response.json()
            
            if not data:
                print(f"   ✓ No more data at page {page}")
                break
            
            for item in data:
                recipe_url = item.get('link', '')
                recipe_title = item.get('title', {}).get('rendered', '')
                if recipe_url:
                    all_urls.append({
                        'url': recipe_url,
                        'title': recipe_title,
                        'id': item.get('id'),
                        'date': item.get('date')
                    })
            
            print(f"   Page {page}: +{len(data)} recipes (Total: {len(all_urls)})")
            page += 1
            
            time.sleep(random.uniform(0.3, 0.8))
            
        except Exception as e:
            print(f"   ✗ Error at page {page}: {e}")
            break
    
    print(f"\n🎉 Total: {len(all_urls)} recipe URLs")
    return all_urls

# Chạy lấy URLs
recipe_urls = get_all_recipe_urls_from_api()

# Lưu
if recipe_urls:
    urls_df = pd.DataFrame(recipe_urls)
    urls_df.to_csv('mngn_recipe_urls.csv', index=False)
    print(f"✅ Saved to mngn_recipe_urls.csv")

📡 Fetching recipe URLs from WordPress API...
   Page 1: +100 recipes (Total: 100)
   Page 1: +100 recipes (Total: 100)
   Page 2: +100 recipes (Total: 200)
   Page 2: +100 recipes (Total: 200)
   Page 3: +100 recipes (Total: 300)
   Page 3: +100 recipes (Total: 300)
   Page 4: +100 recipes (Total: 400)
   Page 4: +100 recipes (Total: 400)
   Page 5: +100 recipes (Total: 500)
   Page 5: +100 recipes (Total: 500)
   Page 6: +100 recipes (Total: 600)
   Page 6: +100 recipes (Total: 600)
   Page 7: +100 recipes (Total: 700)
   Page 7: +100 recipes (Total: 700)
   Page 8: +100 recipes (Total: 800)
   Page 9: +100 recipes (Total: 900)
   Page 10: +100 recipes (Total: 1000)
   Page 11: +100 recipes (Total: 1100)
   Page 12: +100 recipes (Total: 1200)
   Page 13: +100 recipes (Total: 1300)
   Page 14: +100 recipes (Total: 1400)
   Page 15: +100 recipes (Total: 1500)
   Page 16: +100 recipes (Total: 1600)
   Page 17: +100 recipes (Total: 1700)
   Page 18: +100 recipes (Total: 1800)
   Page 19: 

## 2. Crawl chi tiết từng công thức

In [4]:
def get_recipe_detail_selenium(url, driver, category="api"):
    """
    Crawl chi tiết công thức từ Món Ngon Mỗi Ngày sử dụng Selenium
    Website render bằng JavaScript nên cần Selenium để lấy nội dung đã render
    """
    result = {
        "link": url,
        "type_of_food": category,
        "title": None,
        "description": None,
        "author_name": None,
        "cook_time": None,
        "num_of_people": None,
        "calories": None,
        "num_of_ingredients": None,
        "ingredients": [],
        "step": [],
        "note": [],
        "post_date": None,
        "source": "monngonmoingay.com"
    }
    
    def safe_text(element):
        return element.text.strip() if element else None
    
    try:
        driver.get(url)
        # Đợi page load xong (đợi h1 xuất hiện)
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.TAG_NAME, "h1"))
        )
        time.sleep(1.5)  # Đợi thêm để JS render hoàn toàn
        
        # Lấy HTML đã render
        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')
        
        # Title
        h1 = soup.find('h1')
        result['title'] = safe_text(h1) if h1 else None
        
        # Tìm tất cả các h4 (sections)
        all_h4 = soup.find_all('h4')
        
        for h4 in all_h4:
            h4_text = safe_text(h4)
            if not h4_text:
                continue
            
            h4_lower = h4_text.lower()
            
            # Parse NGUYÊN LIỆU - cấu trúc đặc biệt của website này
            if 'nguyên liệu' in h4_lower:
                # Chỉ lấy 1 div chứa nguyên liệu (tránh duplicate)
                sibling = h4.find_next_sibling()
                found_main_div = False
                raw_text = ""
                
                while sibling and sibling.name not in ['h4', 'h3', 'h2']:
                    if sibling.name == 'div':
                        text = sibling.get_text(separator='|', strip=True)
                        # Bỏ qua div ghi chú đơn vị (M: muỗng...)
                        if 'muỗng canh' not in text.lower():
                            if not found_main_div and len(text) > 50:
                                raw_text = text
                                found_main_div = True
                                break  # Chỉ lấy div đầu tiên có nội dung chính
                    sibling = sibling.find_next_sibling()
                
                # Parse raw_text thành ingredients
                if raw_text:
                    parts = re.split(r'[|]+', raw_text)
                    ingredients = []
                    seen = set()
                    
                    i = 0
                    while i < len(parts):
                        part = parts[i].strip()
                        if not part or part in seen:
                            i += 1
                            continue
                        
                        # Kiểm tra xem phần tiếp theo có phải số lượng không
                        if i + 1 < len(parts):
                            next_part = parts[i + 1].strip()
                            # Số lượng thường có số hoặc đơn vị
                            if re.match(r'^\d', next_part) or any(u in next_part.lower() for u in ['gr', 'ml', 'quả', 'ổ', 'lát', 'nhánh', 'củ', 'cây', 'muỗng', 'trái']):
                                merged = f"{part} {next_part}"
                                if merged not in seen:
                                    ingredients.append(merged)
                                    seen.add(merged)
                                i += 2
                                continue
                        
                        # Không có số lượng đi kèm
                        if len(part) > 3 and part not in seen:
                            ingredients.append(part)
                            seen.add(part)
                        i += 1
                    
                    result['ingredients'] = ingredients
                    result['num_of_ingredients'] = len(ingredients)
            
            # Parse SƠ CHẾ
            elif 'sơ chế' in h4_lower and not result['step']:
                sibling = h4.find_next_sibling()
                while sibling and sibling.name not in ['h4', 'h3', 'h2']:
                    if sibling.name:
                        text = sibling.get_text(separator=' ', strip=True)
                        if text and len(text) > 20:
                            items = re.split(r'[•]', text)
                            for item in items:
                                item = item.strip()
                                if item and len(item) > 10:
                                    result['step'].append(f"Sơ chế: {item}")
                            break  # Chỉ lấy element đầu tiên
                    sibling = sibling.find_next_sibling()
            
            # Parse THỰC HIỆN
            elif 'thực hiện' in h4_lower:
                sibling = h4.find_next_sibling()
                while sibling and sibling.name not in ['h4', 'h3', 'h2']:
                    if sibling.name:
                        text = sibling.get_text(separator=' ', strip=True)
                        if text and len(text) > 20:
                            items = re.split(r'[•]', text)
                            for item in items:
                                item = item.strip()
                                if item and len(item) > 10:
                                    result['step'].append(f"Thực hiện: {item}")
                            break
                    sibling = sibling.find_next_sibling()
            
            # Parse MÁCH NHỎ
            elif 'mách nhỏ' in h4_lower and not result['note']:
                sibling = h4.find_next_sibling()
                while sibling and sibling.name not in ['h4', 'h3', 'h2']:
                    if sibling.name:
                        text = sibling.get_text(separator=' ', strip=True)
                        # Bỏ qua table dinh dưỡng
                        if text and not any(x in text for x in ['Bữa sáng', 'Bữa trưa', 'Bữa tối', 'Năng lượng', 'kcal']):
                            if len(text) > 10:
                                items = re.split(r'[•]', text)
                                for item in items:
                                    item = item.strip()
                                    if item and len(item) > 5:
                                        result['note'].append(item)
                                break
                    sibling = sibling.find_next_sibling()
        
        # Lấy thông tin meta từ sidebar
        page_text = soup.get_text()
        
        # Thời gian thực hiện
        time_match = re.search(r'(\d+)\s*Phút', page_text)
        if time_match:
            result['cook_time'] = f"{time_match.group(1)} phút"
        
        # Khẩu phần
        serving_match = re.search(r'(\d+)\s*người', page_text)
        if serving_match:
            result['num_of_people'] = int(serving_match.group(1))
        
        # Description từ meta
        meta_desc = soup.find('meta', attrs={'name': 'description'})
        if meta_desc:
            result['description'] = meta_desc.get('content', '')
        
        return result
        
    except Exception as e:
        print(f"  [ERROR] {url}: {e}")
        return result


def get_recipe_detail_mngn(url, category):
    """
    Fallback function: sử dụng Selenium nếu có
    """
    global driver
    
    if SELENIUM_AVAILABLE and driver:
        return get_recipe_detail_selenium(url, driver, category)
    
    # Fallback: requests (chỉ lấy được title từ meta)
    result = {
        "link": url,
        "type_of_food": category,
        "title": None,
        "description": None,
        "author_name": None,
        "cook_time": None,
        "num_of_people": None,
        "calories": None,
        "num_of_ingredients": None,
        "ingredients": [],
        "step": [],
        "note": [],
        "post_date": None,
        "source": "monngonmoingay.com"
    }
    
    try:
        response = requests.get(url, headers=HEADERS, timeout=30)
        response.encoding = 'utf-8'
        soup = BeautifulSoup(response.text, 'html.parser')
        
        title_tag = soup.find('meta', property='og:title')
        if title_tag:
            result['title'] = title_tag.get('content', '').replace(' - Món Ngon Mỗi Ngày', '')
        
        desc_tag = soup.find('meta', attrs={'name': 'description'})
        if desc_tag:
            result['description'] = desc_tag.get('content', '')
        
        return result
        
    except Exception as e:
        print(f"  [ERROR] {url}: {e}")
        return result

In [5]:
# Khởi tạo driver và test
if SELENIUM_AVAILABLE:
    driver = init_driver()
    
    if driver:
        test_url = "https://monngonmoingay.com/banh-mi-chao/"
        print(f"\n🧪 Testing với: {test_url}")
        
        detail = get_recipe_detail_selenium(test_url, driver, "test")
        
        print(f"\n📋 Kết quả:")
        print(f"  Title: {detail['title']}")
        print(f"  Cook time: {detail['cook_time']}")
        print(f"  Num of people: {detail['num_of_people']}")
        print(f"  Description: {detail['description'][:100] if detail['description'] else 'N/A'}...")
        
        print(f"\n🥗 Ingredients ({len(detail['ingredients'])}):")
        for i, ing in enumerate(detail['ingredients'][:8], 1):
            print(f"  {i}. {ing}")
        if len(detail['ingredients']) > 8:
            print(f"  ... và {len(detail['ingredients']) - 8} nguyên liệu khác")
        
        print(f"\n👨‍🍳 Steps ({len(detail['step'])}):")
        for i, step in enumerate(detail['step'][:3], 1):
            step_display = step[:100] + "..." if len(step) > 100 else step
            print(f"  {i}. {step_display}")
        
        print(f"\n💡 Notes ({len(detail['note'])}):")
        for note in detail['note'][:2]:
            note_display = note[:80] + "..." if len(note) > 80 else note
            print(f"  - {note_display}")
else:
    print("❌ Cần cài Selenium để crawl website này:")
    print("   pip install selenium webdriver-manager")

✅ Chrome driver đã khởi tạo thành công

🧪 Testing với: https://monngonmoingay.com/banh-mi-chao/

📋 Kết quả:
  Title: BÁNH MÌ CHẢO
  Cook time: 15 phút
  Num of people: 2
  Description: N/A...

🥗 Ingredients (5):
  1. Bánh mì ổ		 160gr (2 ổ) Thịt bò phi lê		 150gr
  2. Pate 		 20gr Phô mai lát		 40gr
  3. Trứng gà		 2 quả Xúc xích xông khói 		 100gr
  4. Gia vị: 	Tỏi băm, Hạt nêm Aji-ngon® Heo, Nước tương "Phú Sĩ", Bột ngọt AJI-NO-MOTO®, tiêu, dầu ăn
  5. Xốt Mayonnaise Aji-mayo® Vị Nguyên Bản

👨‍🍳 Steps (2):
  1. Sơ chế: Thịt bò thái lát mỏng ướp với 2m nước tương, 2m dầu ăn, 2m tỏi băm, 1/2m bột ngọt, 1/2m đườn...
  2. Thực hiện: Đun nóng chảo với ít dầu, chiên xúc xích cho thơm giòn rồi lấy ra đĩa, sau đó ta cho thịt...

💡 Notes (1):
  - Ướp thịt bò với ít dầu ăn giúp thịt bò dễ thấm gia vị hơn. Khi xào thì xào với l...

📋 Kết quả:
  Title: BÁNH MÌ CHẢO
  Cook time: 15 phút
  Num of people: 2
  Description: N/A...

🥗 Ingredients (5):
  1. Bánh mì ổ		 160gr (2 ổ) Thịt bò phi lê		 150g

In [6]:
# Test với nhiều URL để đảm bảo parser hoạt động
test_urls = [
    "https://monngonmoingay.com/banh-mi-chao/",
    "https://monngonmoingay.com/com-chien-tom-thom/",
    "https://monngonmoingay.com/sup-ga-bi-do/"
]

for test_url in test_urls:
    print(f"\n{'='*60}")
    print(f"🧪 Testing: {test_url}")
    
    detail = get_recipe_detail_selenium(test_url, driver, "test")
    
    print(f"  ✓ Title: {detail['title']}")
    print(f"  ✓ Cook time: {detail['cook_time']}")
    print(f"  ✓ Ingredients: {len(detail['ingredients'])} items")
    print(f"  ✓ Steps: {len(detail['step'])} items")
    print(f"  ✓ Notes: {len(detail['note'])} items")
    
    if detail['title']:
        print(f"  ✅ SUCCESS")
    else:
        print(f"  ❌ FAILED")
    
    time.sleep(1)


🧪 Testing: https://monngonmoingay.com/banh-mi-chao/
  ✓ Title: BÁNH MÌ CHẢO
  ✓ Cook time: 15 phút
  ✓ Ingredients: 5 items
  ✓ Steps: 2 items
  ✓ Notes: 1 items
  ✅ SUCCESS
  ✓ Title: BÁNH MÌ CHẢO
  ✓ Cook time: 15 phút
  ✓ Ingredients: 5 items
  ✓ Steps: 2 items
  ✓ Notes: 1 items
  ✅ SUCCESS

🧪 Testing: https://monngonmoingay.com/com-chien-tom-thom/

🧪 Testing: https://monngonmoingay.com/com-chien-tom-thom/
  ✓ Title: Cơm chiên tôm thơm
  ✓ Cook time: 10 phút
  ✓ Ingredients: 7 items
  ✓ Steps: 2 items
  ✓ Notes: 1 items
  ✅ SUCCESS
  ✓ Title: Cơm chiên tôm thơm
  ✓ Cook time: 10 phút
  ✓ Ingredients: 7 items
  ✓ Steps: 2 items
  ✓ Notes: 1 items
  ✅ SUCCESS

🧪 Testing: https://monngonmoingay.com/sup-ga-bi-do/

🧪 Testing: https://monngonmoingay.com/sup-ga-bi-do/
  ✓ Title: SÚP GÀ BÍ ĐỎ
  ✓ Cook time: 5 phút
  ✓ Ingredients: 7 items
  ✓ Steps: 2 items
  ✓ Notes: 1 items
  ✅ SUCCESS
  ✓ Title: SÚP GÀ BÍ ĐỎ
  ✓ Cook time: 5 phút
  ✓ Ingredients: 7 items
  ✓ Steps: 2 items
  ✓ Notes: 1

In [7]:
# Load URLs từ file (nếu cần chạy lại)
if os.path.exists('mngn_recipe_urls.csv'):
    urls_df = pd.read_csv('mngn_recipe_urls.csv')
    recipe_urls = urls_df.to_dict('records')  # Chuyển thành list of dicts
    print(f"Loaded {len(recipe_urls)} URLs")

Loaded 2445 URLs


In [8]:
# Crawl chi tiết với Selenium
if not SELENIUM_AVAILABLE or driver is None:
    print("❌ Cần Selenium để crawl. Cài đặt bằng:")
    print("   pip install selenium webdriver-manager")
else:
    all_recipes = []
    failed_urls = []
    CHECKPOINT = 50  # Giảm checkpoint vì Selenium chậm hơn
    
    print(f"📡 Bắt đầu crawl {len(recipe_urls)} recipes...")
    print("⚠️ Selenium sẽ chậm hơn requests (~2-3s/page)")
    
    for i, item in enumerate(tqdm(recipe_urls, desc="Crawling")):
        url = item['url'] if isinstance(item, dict) else item
        
        try:
            detail = get_recipe_detail_selenium(url, driver, "api")
            
            # Điều kiện thành công: có title
            if detail['title']:
                all_recipes.append(detail)
            else:
                failed_urls.append((url, "Missing title"))
            
            # Checkpoint
            if (i + 1) % CHECKPOINT == 0:
                pd.DataFrame(all_recipes).to_csv('mngn_recipes_checkpoint.csv', index=False)
                success_rate = len(all_recipes) / (i + 1) * 100
                print(f"\n  💾 Checkpoint: {len(all_recipes)} recipes ({success_rate:.1f}% success)")
            
            time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))
            
        except Exception as e:
            failed_urls.append((url, str(e)))
    
    print(f"\n✅ Success: {len(all_recipes)}")
    print(f"❌ Failed: {len(failed_urls)}")
    
    # Đóng driver sau khi crawl xong
    # driver.quit()  # Uncomment khi crawl xong

📡 Bắt đầu crawl 2445 recipes...
⚠️ Selenium sẽ chậm hơn requests (~2-3s/page)


Crawling:   1%|▏         | 34/2445 [02:01<2:24:10,  3.59s/it]

  [ERROR] https://monngonmoingay.com/canh-chua-tom-bon-bon-bong-dien-dien/: Message: timeout: Timed out receiving message from renderer: -0.014
  (Session info: chrome=142.0.7444.176)
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x734103
	0x734144
	0x53e71d
	0x52ec5a
	0x52e98d
	0x52c7fe
	0x52d3c7
	0x53a16e
	0x54c095
	0x551be6
	0x52da46
	0x54be27
	0x5ced26
	0x5ac706
	0x57da30
	0x57ed54
	0x9a57b4
	0x9a098a
	0x75c392
	0x74c4c8
	0x75324d
	0x73c478
	0x73c63c
	0x7267ca
	0x75d85d49
	0x7758d6db
	0x7758d661



Crawling:   1%|▏         | 35/2445 [02:32<7:53:36, 11.79s/it]

  [ERROR] https://monngonmoingay.com/hu-tieu-sa-dec-xao-tep-bong-dien-dien/: Message: timeout: Timed out receiving message from renderer: -0.002
  (Session info: chrome=142.0.7444.176)
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x734103
	0x734144
	0x53e71d
	0x52ec5a
	0x52e98d
	0x52c7fe
	0x52d3c7
	0x53a16e
	0x54c095
	0x551be6
	0x52da46
	0x54be27
	0x5ced26
	0x5ac706
	0x57da30
	0x57ed54
	0x9a57b4
	0x9a098a
	0x75c392
	0x74c4c8
	0x75324d
	0x73c478
	0x73c63c
	0x7267ca
	0x75d85d49
	0x7758d6db
	0x7758d661



...

  [ERROR] https://monngonmoingay.com/ca-thu-rim-nuoc-tra-xanh/: Message: timeout: Timed out receiving message from renderer: -0.005
  (Session info: chrome=142.0.7444.176)
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x734103
	0x734144
	0x53e71d
	0x52ec5a
	0x52e98d
	0x52c7fe
	0x52d3c7
	0x53a16e
	0x54c095
	0x551be6
	0x52da46
	0x54be27
	0x5ced26
	0x5ac706
	0x57da30
	0x57ed54
	0x9a57b4
	0x9a098a
	0x75c392
	0x74c4c8
	0x75324d
	0x73c478
	0x73c63c
	0x7267ca
	0x75d85d49
	0x7758d6db
	0x7758d661



Crawling: 100%|█████████▉| 2435/2445 [2:35:48<02:03, 12.36s/it]

  [ERROR] https://monngonmoingay.com/ca-hu-kho-xot-tuong/: Message: timeout: Timed out receiving message from renderer: -0.004
  (Session info: chrome=142.0.7444.176)
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x734103
	0x734144
	0x53e71d
	0x52ec5a
	0x52e98d
	0x52c7fe
	0x52d3c7
	0x53a16e
	0x54c095
	0x551be6
	0x52da46
	0x54be27
	0x5ced26
	0x5ac706
	0x57da30
	0x57ed54
	0x9a57b4
	0x9a098a
	0x75c392
	0x74c4c8
	0x75324d
	0x73c478
	0x73c63c
	0x7267ca
	0x75d85d49
	0x7758d6db
	0x7758d661



Crawling: 100%|██████████| 2445/2445 [2:36:53<00:00,  3.85s/it]


✅ Success: 2426
❌ Failed: 19


In [9]:
# Lưu kết quả
mngn_df = pd.DataFrame(all_recipes)
mngn_df.to_csv('mngn_recipes_detail.csv', index=False)
print(f"Saved {len(mngn_df)} recipes to mngn_recipes_detail.csv")

if failed_urls:
    pd.DataFrame(failed_urls, columns=['url', 'error']).to_csv('mngn_failed_urls.csv', index=False)
    print(f"Saved {len(failed_urls)} failed URLs")

Saved 2426 recipes to mngn_recipes_detail.csv
Saved 19 failed URLs


## 3. Merge tất cả nguồn dữ liệu

In [10]:
# Load tất cả sources
dataframes = []

# VnExpress
if os.path.exists('vnexpress_foods_detail_merged.csv'):
    vne_df = pd.read_csv('vnexpress_foods_detail_merged.csv')
    vne_df['source'] = 'vnexpress.net'
    dataframes.append(vne_df)
    print(f"VnExpress: {len(vne_df)} recipes")

# Món Ngon Mỗi Ngày
if os.path.exists('mngn_recipes_detail.csv'):
    mngn_df = pd.read_csv('mngn_recipes_detail.csv')
    dataframes.append(mngn_df)
    print(f"Món Ngon Mỗi Ngày: {len(mngn_df)} recipes")

# DMX (nếu có)
if os.path.exists('dmx_recipes_detail.csv'):
    dmx_df = pd.read_csv('dmx_recipes_detail.csv')
    if len(dmx_df) > 0:
        dataframes.append(dmx_df)
        print(f"Điện Máy Xanh: {len(dmx_df)} recipes")

VnExpress: 893 recipes
Món Ngon Mỗi Ngày: 2426 recipes
Điện Máy Xanh: 252 recipes


In [11]:
# Chuẩn hóa cột
common_cols = [
    'link', 'type_of_food', 'title', 'description', 'author_name',
    'cook_time', 'num_of_people', 'calories', 'num_of_ingredients',
    'ingredients', 'step', 'note', 'post_date', 'source'
]

for i, df in enumerate(dataframes):
    for col in common_cols:
        if col not in df.columns:
            df[col] = None
    dataframes[i] = df[common_cols]

In [12]:
# Merge
if dataframes:
    merged_df = pd.concat(dataframes, ignore_index=True)
    merged_df = merged_df.drop_duplicates(subset=['title'], keep='first')
    
    print(f"\n📊 Final Merged Dataset:")
    print(f"  Total: {len(merged_df)} recipes")
    print(f"\n  By source:")
    print(merged_df['source'].value_counts())
    
    # Lưu
    merged_df.to_csv('all_recipes_final.csv', index=False)
    print(f"\n✅ Saved to all_recipes_final.csv")


📊 Final Merged Dataset:
  Total: 3524 recipes

  By source:
source
monngonmoingay.com    2386
vnexpress.net          886
dienmayxanh.com        252
Name: count, dtype: int64

✅ Saved to all_recipes_final.csv


In [13]:
# Thống kê cuối cùng
print("📈 Thống kê loại món:")
print(merged_df['type_of_food'].value_counts().head(20))

📈 Thống kê loại món:
type_of_food
api                            2386
Món ngon hàng ngày              486
Món ngon cho cuối tuần          123
Món Tết                          81
Món tráng miệng, giải khát       56
Quà - Món ăn vặt                 51
Món Chay                         24
Món ngon theo vùng miền          24
Món Chè                          23
Món Nướng                        22
Món Hấp                          22
Món Gỏi                          21
Món ngon ngày lạnh               21
Món Kho                          20
Món Chiên                        20
Món Xào                          20
Thực đơn cho ngày nắng nóng      19
Món Lẩu                          19
Món Bánh                         18
Món Thịt Heo                     16
Name: count, dtype: int64
